[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [2]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [3]:
import torch
import math

In [10]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    B,seq,D=q.shape

    dim_indices=torch.arange(0,D,2)
    inv_freq=1/(10000**(dim_indices/D))

    pos=torch.arange(seq)
    angles=pos.unsqueeze(1)*inv_freq.unsqueeze(0)

    angles_ext=torch.repeat_interleave(angles,repeats=2,dim=-1)
    cos=angles_ext.view(1,seq,D).cos()
    sin=angles_ext.view(1,seq,D).sin()

    def rotate_half(x):
      x_even=x[...,0::2]
      x_odd=x[...,1::2]

      return torch.stack((-x_odd,x_even),dim=-1).flatten(-2)
    q_out=(q*cos)+(rotate_half(q))*sin
    k_out=(k*cos)+(rotate_half(k) * sin)

    return q_out,k_out




In [11]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [12]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (6.5ms)
  ✅ [2/4] Preserves norm (10.1ms)
  ✅ [3/4] Relative position property (18.5ms)
  ✅ [4/4] Gradient flow (45.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (80.3ms total)
  Progress saved. Run status() to see your dashboard.

